# 09 · Preparación de los datos de la app

Ejecutar de arriba abajo después del notebook 08, con un kernel limpio. Este notebook carga sus propias variables; no necesita memoria de otros notebooks.

**Entradas:** `data/processed/sitewise_v03/valores.csv`, `data/processed/sitewise_v03/dataset_municipal.csv` (03) y `data/processed/sitewise_ml/agua_estimada_vecinos_v01.csv` (sección 1 del 08).

**Salidas:** `valores_app.csv`, `dataset_municipal_app.csv` y la misma correspondencia de estados en `data/processed/sitewise_v03/`.

Opción A: incorporar los 722 valores estimados exclusivamente al indicador cuantitativo de la app, con estado aproximado, método y distancias. Los originales y el indicador químico permanecen intactos. No hay que regresar al notebook 03.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Abre el notebook dentro de la raíz del repositorio o de su carpeta notebooks.")
ruta = ROOT / "data/processed/sitewise_v03/valores.csv"
for archivo in [ruta, ruta.parent / "dataset_municipal.csv"]:
    if not archivo.is_file():
        raise FileNotFoundError(f"Falta {archivo.relative_to(ROOT)}. Ejecuta primero el notebook 03.")
valores = pd.read_csv(ruta, dtype={"cod_ine": str}, low_memory=False)
print("Datos originales cargados:", len(valores), "registros")


## Integración y validación

El CSV contiene la media de los cinco municipios más cercanos con dato, por distancia haversine entre coordenadas. Solo se rellenan ausencias del indicador cuantitativo con estado original `sin_masa_catalogada`. El químico no se estima.

Se conserva el estado de origen en `estado`; la interfaz debe usar `estado_app`. Se conservan valor, método y naturaleza originales, y se identifican las filas estimadas. El método de vecinos no acredita caudal ni suministro. El error medio del 08 no garantiza el error individual de los 722 municipios ni constituye una validación espacial independiente. No se introducen nuevos umbrales de distancia.


In [ ]:
# Opción A: estimaciones exclusivamente en los CSV de la app.
# Las celdas anteriores de este notebook cargan todos los datos necesarios.
import hashlib

IND_AGUA = "agua_subterranea_mal_estado_cuantitativo_pct"
carpeta = ruta.parent
ruta_estimaciones = ROOT / "data/processed/sitewise_ml/agua_estimada_vecinos_v01.csv"
if not ruta_estimaciones.is_file():
    raise FileNotFoundError(f"Falta {ruta_estimaciones}. Ejecutar primero la sección 1 del notebook 08 (08_mlflow_segunda_prueba.ipynb).")

# Controlar también los bytes originales antes y después de exportar.
originales = [carpeta / "valores.csv", carpeta / "dataset_municipal.csv"]
huellas = {p: hashlib.sha256(p.read_bytes()).hexdigest() for p in originales}
estimaciones = pd.read_csv(ruta_estimaciones, dtype={"cod_ine": str})
requeridas = {"cod_ine", "valor_estimado_pct", "k_vecinos", "distancia_media_km",
              "distancia_maxima_km", "estado_propuesto", "metodo"}
if not requeridas.issubset(estimaciones.columns):
    raise ValueError(f"Faltan columnas: {sorted(requeridas - set(estimaciones.columns))}")
if (len(estimaciones) != 722 or not estimaciones.cod_ine.is_unique
        or not estimaciones.cod_ine.str.fullmatch(r"\d{5}").fillna(False).all()):
    raise ValueError("Se esperan 722 códigos INE únicos de cinco cifras.")
for c in ["valor_estimado_pct", "k_vecinos", "distancia_media_km", "distancia_maxima_km"]:
    estimaciones[c] = pd.to_numeric(estimaciones[c], errors="raise")
    if not np.isfinite(estimaciones[c]).all():
        raise ValueError(f"Valores no finitos en {c}")
if not (estimaciones.valor_estimado_pct.between(0, 100).all()
        and estimaciones.k_vecinos.eq(5).all()
        and estimaciones.distancia_media_km.ge(0).all()
        and estimaciones.distancia_maxima_km.ge(estimaciones.distancia_media_km).all()
        and estimaciones.estado_propuesto.eq("aproximado").all()
        and estimaciones.metodo.fillna("").str.strip().ne("").all()):
    raise ValueError("Estimaciones, estados o metadatos fuera del formato revisado.")
if valores.duplicated(["cod_ine", "id_indicador"]).any():
    raise ValueError("Hay registros originales duplicados.")
objetivo = valores.id_indicador.eq(IND_AGUA) & valores.estado.eq("sin_masa_catalogada")
if (objetivo.sum() != 722 or not valores.loc[objetivo, "valor"].isna().all()
        or set(valores.loc[objetivo, "cod_ine"]) != set(estimaciones.cod_ine)):
    raise ValueError("El CSV no coincide exactamente con las 722 ausencias originales del indicador cuantitativo.")

grupos = {
    "real": ["real", "publicado", "derivado_mismos_meses", "agregado_fusion_documentada"],
    "aproximado": ["aproximado", "aproximado_modelo", "calculado_sobre_masas_catalogadas", "sin_poligonos_en_catalogo"],
    "pendiente": ["pendiente", "ocultado_menor5", "blanco_fuente", "sin_datos_comparables",
                  "sin_masa_catalogada", "sin_destino_catalogado_en_tierra", "base_cero",
                  "sin_cobertura_suficiente", "fusion_con_celdas_no_publicadas", "cobertura_insuficiente"],
}
correspondencia = {original: simple for simple, originales_grupo in grupos.items() for original in originales_grupo}
valores_app = valores.copy(deep=True)
valores_app["estado_app"] = valores_app.estado.map(correspondencia)
if valores_app.estado_app.isna().any():
    raise ValueError("Estados sin correspondencia: " + repr(valores_app.loc[valores_app.estado_app.isna(), "estado"].unique().tolist()))

# Mantener el estado original; la app utiliza estado_app. Conservar valor y metodología de origen.
for c in ["valor", "metodo", "naturaleza"]:
    valores_app[c + "_original"] = valores[c]
valores_app["estimado_app"] = False
por_codigo = estimaciones.set_index("cod_ine")
codigos = valores_app.loc[objetivo, "cod_ine"]
valores_app.loc[objetivo, "valor"] = codigos.map(por_codigo.valor_estimado_pct).to_numpy()
valores_app.loc[objetivo, "estado_app"] = "aproximado"
valores_app.loc[objetivo, "metodo"] = codigos.map(por_codigo.metodo).to_numpy()
valores_app.loc[objetivo, "naturaleza"] = "estimado_vecinos_geograficos"
valores_app.loc[objetivo, "estimado_app"] = True
for c in ["k_vecinos", "distancia_media_km", "distancia_maxima_km"]:
    valores_app[c] = np.nan
    valores_app.loc[objetivo, c] = codigos.map(por_codigo[c]).to_numpy()
valores_app["fuente_estimacion"] = ""
valores_app.loc[objetivo, "fuente_estimacion"] = ruta_estimaciones.relative_to(ROOT).as_posix()

if (valores_app.estado_app.ne("pendiente") & valores_app.valor.isna()).any():
    raise ValueError("Hay estados utilizables sin valor.")
pd.testing.assert_frame_equal(valores_app.loc[~objetivo, valores.columns], valores.loc[~objetivo])
sin_cambios = [c for c in valores.columns if c not in ["valor", "metodo", "naturaleza"]]
pd.testing.assert_frame_equal(valores_app[sin_cambios], valores[sin_cambios])

dataset_original = pd.read_csv(carpeta / "dataset_municipal.csv", dtype={"cod_ine": str, "cod_provincia": str}, low_memory=False)
if not dataset_original.cod_ine.is_unique or set(dataset_original.cod_ine) != set(valores.cod_ine):
    raise ValueError("Municipios incoherentes entre el formato largo y el ancho.")
dataset_app = dataset_original.copy(deep=True)
destino = dataset_app.cod_ine.isin(estimaciones.cod_ine)
if not dataset_app.loc[destino, IND_AGUA].isna().all():
    raise ValueError("El dataset ancho contiene valores que se sobrescribirían.")
dataset_app["valor_original__" + IND_AGUA] = dataset_original[IND_AGUA]
dataset_app.loc[destino, IND_AGUA] = dataset_app.loc[destino, "cod_ine"].map(por_codigo.valor_estimado_pct)
estados_app = valores_app.pivot(index="cod_ine", columns="id_indicador", values="estado_app").add_prefix("estado_app__")
dataset_app = dataset_app.merge(estados_app, on="cod_ine", how="left", validate="one_to_one")
dataset_app["estimado_app__" + IND_AGUA] = dataset_app.cod_ine.isin(estimaciones.cod_ine)
for c in ["metodo", "k_vecinos", "distancia_media_km", "distancia_maxima_km"]:
    dataset_app[c + "_estimacion__" + IND_AGUA] = dataset_app.cod_ine.map(por_codigo[c])

# Coherencia entre las dos salidas y conservación de todos los demás indicadores.
columnas_intactas = [c for c in dataset_original.columns if c != IND_AGUA]
pd.testing.assert_frame_equal(dataset_app[columnas_intactas], dataset_original[columnas_intactas])
pd.testing.assert_series_equal(dataset_app.loc[~destino, IND_AGUA], dataset_original.loc[~destino, IND_AGUA])
largo = valores_app.loc[valores_app.id_indicador.eq(IND_AGUA)].set_index("cod_ine")
np.testing.assert_allclose(dataset_app[IND_AGUA], dataset_app.cod_ine.map(largo.valor), equal_nan=True)
assert dataset_app["estado_app__" + IND_AGUA].eq(dataset_app.cod_ine.map(largo.estado_app)).all()
assert valores_app.estimado_app.sum() == dataset_app["estimado_app__" + IND_AGUA].sum() == 722

valores_app.to_csv(carpeta / "valores_app.csv", index=False, encoding="utf-8")
dataset_app.to_csv(carpeta / "dataset_municipal_app.csv", index=False, encoding="utf-8")
# correspondencia_estados.csv conserva exactamente la misma correspondencia.
pd.DataFrame(correspondencia.items(), columns=["estado_original", "estado_app"]).to_csv(
    carpeta / "correspondencia_estados.csv", index=False, encoding="utf-8")
assert all(hashlib.sha256(p.read_bytes()).hexdigest() == huellas[p] for p in originales)
print("Opción A validada: 722 estimaciones cuantitativas en la app; originales intactos.")
print("El indicador químico conserva sus ausencias. La estimación no acredita caudal ni suministro.")
display(valores_app.estado_app.value_counts())
print("Archivos guardados en:", carpeta)
